# Single spin Part 3: Single qubit gate optimisation using GRAPE

#### 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cthree.signal.envelopes import GaussEnvelope
from cthree.signal.pwc_generator import PWCGenerator
from cthree.signal.waveform import DRAGMixer

#### 1. Define Hamiltonian in the rotating frame of drive

Next, we setup the qubit system we want to control. We define the Hamiltonian in the rotating frame of drive such that the pulse oscillates slowly to apply GRAPE gradients.

The Hamiltonain in the rotating frame of the drive is given by - 
$$ H(t) = \big(\omega_q - \omega_d\big) b^\dagger b -\frac{\alpha}{2} (b^\dagger)^2 b^2 + (\epsilon(t) b + \epsilon(t)^* b)$$

In [ ]:
from cthree.quantity import Quantity
from cthree.model.closed_system import ClosedSystem
from cthree.model.rotating_frame_drive import RotatingFrameDrive
from cthree.model.transmon import Transmon


FREQ = 7.86e9 * 2 * np.pi
DIMS = 3
ANHARM = -50e6 * 2 * np.pi
offset = 5e6 * 2 * np.pi

drive_freq = FREQ + offset
qubit_freq = FREQ - drive_freq

First, let's generate a piecewise constant (PWC) pulse envelope for the Gaussian pulse

In [ ]:
t_final = 25e-9
tlist = np.linspace(0, t_final, 101)
tone = GaussEnvelope(amplitude=Quantity(3 * np.pi / t_final / 3, -20 * np.pi / t_final, 20 * np.pi / t_final))
tone.t_final.set_value(t_final)

drag_tone = DRAGMixer(
    tone,
    deltas=[Quantity(2 * ANHARM, min_value=3 * ANHARM, max_value=ANHARM / 3, unit="Hz", two_pi=True, name="Delta")],
)

gen = PWCGenerator(envelopes=[drag_tone], tlist=tlist)
gen.multiply_flat_top = True

In [ ]:
params = drag_tone.get_parameters()
params

In [ ]:
ts = np.linspace(0, t_final, 1001)

plt.plot(ts / 1e-9, np.real(drag_tone.compute_output(ts)) / 1e6, label="Smooth curve")
plt.plot(ts / 1e-9, np.imag(drag_tone.compute_output(ts)) / 1e6, label="Smooth curve")
plt.plot(ts / 1e-9, np.real(gen.generate_signal(ts)) / 1e6, ls="--", label="Inphase")
plt.plot(
    ts / 1e-9,
    np.imag(gen.generate_signal(ts)) / 1e6,
    ls="--",
    label="Quadrature",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

In [ ]:
Drive = RotatingFrameDrive(gen)
transmon = Transmon(
    frequency=Quantity(
        qubit_freq,
        1.2 * qubit_freq,
        0.8 * qubit_freq,
        unit="Hz",
        name="Frequency",
    ),
    anharmonicity=Quantity(ANHARM, 1.2 * ANHARM, 0.8 * ANHARM, unit="Hz", name="Anharmonicity"),
    drives=[Drive],
    dimension=DIMS,
)

model = ClosedSystem(transmon)

In [ ]:
from cthree.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from cthree.propagation.scipy_expm_grape import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=1e9)

init = np.array([[1.0], [0.0], [0]])  # |0>
target = np.array([[0.0], [1.0], [0]])  # |1>

prop.set_initial_state(init)
prop.target_state = target

prop.use_schirmer_derivative = True

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initial_state=init,
    target_state=target,
    times=tlist,
)

In [ ]:
def plotStates():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generate_signal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


plotStates()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
zeroone.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone and optimise just amplitude and frequency, as in the state transfer example.

In [ ]:
from cthree.optimisation_map import OptimisationMap
from cthree.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient
from cthree.optimisers.scipy_optimiser import ScipyOptimiser
from cthree.measurement.goat_over_grape import GOATOverGRAPE


optmap = OptimisationMap()
# optmap.add(drag_tone, [params[0]])
optmap.add(drag_tone, [params[2]])
# optmap.add(transmon, [transmon.getParameters()[2]])
optmap.register_params_with_optimisables()

goat = GOATOverGRAPE(zeroone, gen)

opt = ScipyOptimiser(goat, optimisables=optmap)
optgrad = ScipyOptimiserGradient(goat, optimisables=optmap)

In [ ]:
optmap

In [ ]:
optgrad.optimise()

In [ ]:
opt.optimise()

In [ ]:
optgrad.optimise()

In [ ]:
ts = np.linspace(0, t_final, 1001)

plt.plot(ts / 1e-9, np.real(gen.generate_signal(ts)) / 1e6, ls="--", label="Inphase")
plt.plot(
    ts / 1e-9,
    np.imag(gen.generate_signal(ts)) / 1e6,
    ls="--",
    label="Quadrature",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

In [ ]:
plotStates()